# Fine-tuning d'un transformer pour le sentiment en derja tunisien

Données : TSAC (commentaires Facebook en derja, écriture arabe et arabizi), nettoyées et découpées par `prepare_data.py`.

**Avant de lancer :** *Exécution › Modifier le type d'exécution › GPU (T4)*, puis importer `train.csv`, `val.csv`, `test.csv` (dossier `ml/data/`) avec l'icône 📁 à gauche.

Baseline à battre (TF-IDF + régression logistique) : **accuracy 0,951 / F1 macro 0,951** sur le test.

In [ ]:
!pip -q install transformers datasets evaluate accelerate

In [ ]:
# Modèle à fine-tuner. xlm-roberta-base gère l'écriture arabe ET latine (arabizi).
# Autre essai possible : "UBC-NLP/MARBERTv2" (dialectes arabes, mais faible en arabizi).
MODEL_NAME = "xlm-roberta-base"
MAX_LEN = 128
EPOCHS = 3
SEED = 42

In [ ]:
import re, unicodedata, numpy as np, pandas as pd, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
set_seed(SEED)
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun (active le GPU !)")

def normalize(text):
    s = unicodedata.normalize("NFKC", str(text)).lower()
    s = re.sub(r"https?://\S+", " ", s)
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)
    s = re.sub(r"[إأآ]", "ا", s).replace("ة", "ه").replace("ى", "ي")
    s = re.sub(r"[\u064B-\u0652]", "", s)
    return re.sub(r"\s+", " ", s).strip()

def is_arabic(text):
    return bool(re.search(r"[\u0600-\u06FF]", unicodedata.normalize("NFKC", str(text))))

train_df, val_df, test_df = (pd.read_csv(f) for f in ["train.csv", "val.csv", "test.csv"])
print(len(train_df), len(val_df), len(test_df))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_dataset(df):
    ds = Dataset.from_dict({"text": [normalize(t) for t in df.text], "label": df.label.tolist()})
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN), batched=True)

train_ds, val_ds, test_ds = map(to_dataset, [train_df, val_df, test_df])

In [ ]:
id2label = {0: "négatif", 1: "positif"}
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id={v: k for k, v in id2label.items()})

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, preds), "f1_macro": f1_score(p.label_ids, preds, average="macro")}

args = TrainingArguments(
    output_dir="out", num_train_epochs=EPOCHS, learning_rate=2e-5,
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    warmup_ratio=0.1, weight_decay=0.01, fp16=torch.cuda.is_available(),
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1_macro", save_total_limit=1, report_to="none", seed=SEED)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
trainer.train()

## Évaluation sur le test (jamais vu pendant l'entraînement)

In [ ]:
pred = trainer.predict(test_ds).predictions.argmax(-1)
y = test_df.label.values
print(classification_report(y, pred, target_names=["négatif", "positif"], digits=4))
print("Matrice de confusion :\n", confusion_matrix(y, pred))
arabic = test_df.text.map(is_arabic).values
for name, m in [("arabe", arabic), ("arabizi", ~arabic)]:
    print(f"{name:8s} accuracy {accuracy_score(y[m], pred[m]):.4f}  F1 {f1_score(y[m], pred[m], average='macro'):.4f}")

In [ ]:
# Erreurs à analyser (sarcasme, mélange de langues, commentaires très courts…)
errors = test_df[pred != y].assign(pred=pred[pred != y])
errors.head(30)

## Export du modèle pour le service Python

In [ ]:
import json, shutil
trainer.save_model("transformer")
tokenizer.save_pretrained("transformer")
json.dump({"model": MODEL_NAME,
           "test": {"accuracy": round(accuracy_score(y, pred), 4), "f1_macro": round(f1_score(y, pred, average="macro"), 4)}},
          open("transformer/metrics.json", "w"), indent=2)
shutil.make_archive("transformer", "zip", "transformer")
from google.colab import files
files.download("transformer.zip")

Décompresse `transformer.zip` dans `ml/models/transformer/`. Le service Python l'utilisera automatiquement à la place de la baseline.